<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/dpr_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>
<font color="#2D2926">CS 4782 Final Project: Dense Passage Retriever Evaluation</font>
</h1>

<hr color="#677BAB" size="4">

<h2>
<font color="#677BAB">Summary</font>
</h2>

<font color="#2D2926">
This notebook evaluates the retrieval component of a small-scale RAG-style open-domain question answering pipeline. It uses Dense Passage Retrieval to encode questions and candidate passages into a shared vector space, then retrieves the top-k passages using FAISS maximum inner product search.
</font>

<h2>
<font color="#677BAB">Role in the Project</font>
</h2>

<font color="#2D2926">
This notebook isolates the non-parametric memory component of RAG. Instead of generating answers directly, it tests whether DPR retrieves passages that contain or overlap with the gold answer. The results help explain the downstream behavior of the RAG-Sequence and RAG-Token notebooks.
</font>

<h2>
<font color="#677BAB">Implementation Details</font>
</h2>

<font color="#2D2926">
<ul>
  <li>Question encoder: <code>facebook/dpr-question_encoder-single-nq-base</code></li>
  <li>Context encoder: <code>facebook/dpr-ctx_encoder-single-nq-base</code></li>
  <li>Retrieval index: FAISS <code>IndexFlatIP</code></li>
  <li>Dataset: Natural Questions subset from Hugging Face</li>
  <li>Split: 10,000 training examples and 1,000 evaluation examples</li>
  <li>Passage corpus: 10,000 / 87,599 SQuAD training contexts</li>
  <li>Metrics: token-level F1 and answer Recall@10</li>
  <li>Output: <code>results/tables/dpr_results.csv</code></li>
</ul>
</font>

<h2>
<font color="#677BAB">Notes and Limitations</font>
</h2>

<font color="#2D2926">
This notebook is not a full Wikipedia-scale DPR reproduction.
The original RAG setup retrieves from a large Wikipedia index, while this project uses a smaller SQuAD context subset for computational feasibility.
<br><br>

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
import re
import torch
import faiss
import numpy as np
import pandas as pd

import os
os.makedirs("results/tables", exist_ok=True)

from datasets import load_dataset
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    DPRContextEncoder,
    DPRContextEncoderTokenizer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

Device set to cuda


# NQ dataset

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train")
split_dataset = nq.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"].select(range(10000))
eval_dataset = split_dataset["test"].select(range(1000))

dataset_name = "natural-questions"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

# Helper functions

In [ ]:
def get_question(example):
    return example["query"]

def get_answer(example):
    answer = example["answer"]
    if isinstance(answer, list):
        answer = answer[0] if len(answer) > 0 else ""
    return answer

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Token-level F1

In [ ]:
def qa_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = set(pred_tokens) & set(gold_tokens)
    num_same = sum(
        min(pred_tokens.count(tok), gold_tokens.count(tok))
        for tok in common
    )

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

# Extra Metric for DPR (answer recall@10)

In [ ]:
def answer_in_docs(answer, docs):
    answer = normalize_text(answer)
    docs_text = normalize_text(" ".join(docs))
    return int(answer in docs_text)

# Build Passage Corpus

In [ ]:
squad_contexts = load_dataset("squad", split="train[:10000]")

def build_passage_corpus_from_contexts(dataset):
    passages = []

    for i in range(len(dataset)):
        context = str(dataset[i]["context"]).strip()

        if context != "":
            passages.append(context)

    passages = list(dict.fromkeys(passages))
    return passages

passages = build_passage_corpus_from_contexts(squad_contexts)

print("Number of SQuAD passages:", len(passages))
print(passages[0])

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Number of SQuAD passages: 1867
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.


# Load DPR models

In [ ]:
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = question_encoder.to(device)


context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = context_encoder.to(device)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Question Encoding

In [ ]:
def encode_question(question):
  inputs = question_tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = question_encoder(**inputs).pooler_output
  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Passage Encoding

In [ ]:
def encode_passage(passage):
  inputs = context_tokenizer(passage, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = context_encoder(**inputs).pooler_output

  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Build FAISS index

In [ ]:
def build_faiss_index(passages):
  passages_embeddings = np.stack([encode_passage(passage) for passage in passages]).astype(np.float32)
  faiss.normalize_L2(passages_embeddings)
  index = faiss.IndexFlatIP(passages_embeddings.shape[1])
  index.add(passages_embeddings)
  return index

index = build_faiss_index(passages)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

# Retrieval function

In [ ]:
def retrieval_top_k(question, index, passages, k = 10):
  question_embedding = encode_question(question)
  question_embedding = question_embedding.reshape(1, -1)
  faiss.normalize_L2(question_embedding)
  distances, indices = index.search(question_embedding, k)
  relevant_passages = [passages[i] for i in indices[0]]
  return relevant_passages, distances[0]

# Prediction rule

In [ ]:
def dpr_prediction_rule(question, index, passages, k = 10):
  relevant_passages, distances = retrieval_top_k(question, index, passages, k)
  prediction = relevant_passages[0]
  return prediction, relevant_passages

# Smoke Test

In [ ]:
question = get_question(eval_dataset[0])
answer = get_answer(eval_dataset[0])

prediction, relevant_passages = dpr_prediction_rule(question, index, passages)

print("Question: ", question)
print("Answer: ", answer)
print("Prediction: ", prediction)
print("Relevant passages: ", relevant_passages)

Question:  who are the basques and where do they live
Answer:  Basques The Basques (/bɑːsks/ or /bæsks/; Basque: euskaldunak [eus̺kaldunak]; Spanish: vascos [ˈbaskos]; French: basques [bask]) are an indigenous ethnic group[5][6][7] characterised by the Basque language, a common culture and shared ancestry to the ancient Vascones and Aquitanians.[8] Basques are indigenous to and primarily inhabit an area traditionally known as the Basque Country (Basque: Euskal Herria), a region that is located around the western end of the Pyrenees on the coast of the Bay of Biscay and straddles parts of north-central Spain and south-western France.
Prediction:  The Statistics Portugal (Portuguese: INE - Instituto Nacional de Estatística) estimates that, according to the 2011 census, the population was 10,562,178 (of which 52% was female, 48% was male). This population has been relatively homogeneous for most of its history: a single religion (Catholicism) and a single language have contributed to this

# Evaluation

In [ ]:
results = []

for i in range(len(eval_dataset)):
  question = get_question(eval_dataset[i])
  answer = get_answer(eval_dataset[i])
  prediction, relevant_passages = dpr_prediction_rule(question, index, passages)
  f1 = qa_f1(prediction, answer)
  recall_at_10 = answer_in_docs(answer, relevant_passages)

  results.append({
      "dataset": dataset_name,
      "question_number": i,
      "question": question,
      "answer": answer,
      "prediction": prediction,
      "f1": f1,
      "recall_at_10": recall_at_10,
      "relevant_passages": relevant_passages
  })

results_df = pd.DataFrame(results)
results_df.head()

,dataset,question_number,question,answer,prediction,f1,recall_at_10,relevant_passages
0,natural-questions,0,who are the basques and where do they live,Basques The Basques (/bɑːsks/ or /bæsks/; Basq...,The Statistics Portugal (Portuguese: INE - Ins...,0.156522,0,[The Statistics Portugal (Portuguese: INE - In...
1,natural-questions,1,where does the name led zeppelin come from,Led Zeppelin The band completed the Scandinavi...,"The term was created in 1920 by Hans Winkler, ...",0.121212,0,"[The term was created in 1920 by Hans Winkler,..."
2,natural-questions,2,where is the heart located in the rib cage,Thorax The anatomy of the chest can also be de...,Domestic dogs have been selectively bred for m...,0.065574,0,[Domestic dogs have been selectively bred for ...
3,natural-questions,3,who sings everybody in the club get tipsy,"Tipsy (song) ""Tipsy"" is a record by American r...","Kanye Omari West (/ˈkɑːnjeɪ/; born June 8, 197...",0.151899,0,"[Kanye Omari West (/ˈkɑːnjeɪ/; born June 8, 19..."
4,natural-questions,4,how many mg of thc to test positive,Cannabis drug testing The main metabolite excr...,Solar water disinfection (SODIS) involves expo...,0.092784,0,[Solar water disinfection (SODIS) involves exp...


In [ ]:
final_f1 = results_df["f1"].mean()*100
print("DPR F1: ", final_f1)
print("DPR Recall@10:", results_df["recall_at_10"].mean() * 100)
results_df.to_csv("results/tables/dpr_results.csv", index=False)
print("Saved dpr_results.csv")

DPR F1:  13.726270413908548
DPR Recall@10: 0.0
Saved dpr_results.csv


<h2>
<font color="#677BAB">Final Interpretation</font>
</h2>

<font color="#2D2926">
This notebook is part of a constrained re-implementation of Retrieval-Augmented Generation for open-domain question answering. The main limitation is scale. The results are therefore most useful for comparing relative behavior across the T5 baseline, DPR retriever, RAG-Sequence, and RAG-Token systems rather than claiming full reproduction of the original paper’s reported numbers.
</font>